## time列の追加

In [ ]:
import pandas as pd
import os

# データ読み込み
df = pd.read_csv("../data/raw/train.csv", encoding='cp932')

# 各樹種ごとにsample numberを1始まりで振り直したtime列を追加
df['time'] = df.groupby('species number')['sample number'].rank(method='first').astype(int)

# time列をsample number, species numberの直後に移動
cols = df.columns.tolist()
cols.insert(2, cols.pop(cols.index('time')))
df = df[cols]

# 保存
os.makedirs("../data/middle", exist_ok=True)
save_path = "../data/middle/train_with_time.csv"
df.to_csv(save_path, index=False, encoding='cp932')

print(f"shape: {df.shape}")
print(f"保存完了: {save_path}")
print("\ntime範囲（樹種別）:")
print(df.groupby('species number')['time'].agg(['min', 'max', 'count']))
df.head()


## binごとにスペクトルを分けるデータ

In [4]:
import pandas as pd
import numpy as np
import os

# ===== パラメータ =====
bin_width = 1000   # ビン幅 (cm⁻¹)
# ====================

# データ読み込み
df = pd.read_csv("../data/middle/train_with_time.csv", encoding='cp932')

meta_cols = ['sample number', 'time', 'species number', '樹種', '含水率']
spec_cols = [c for c in df.columns if c not in meta_cols]
spec_cols_float = [(c, float(c)) for c in spec_cols if float(c) >= 4000.0]

# ビニング
bin_starts = np.arange(4000, 10000, bin_width)
bins = np.arange(4000, 10000 + bin_width, bin_width)

bin_col_map = {}
for i in range(len(bins) - 1):
    low, high = bins[i], bins[i + 1]
    label = f"{int(low)}-{int(high)}"
    matched = [c for c, v in spec_cols_float if low <= v < high]
    if matched:
        bin_col_map[label] = matched

df_out = df[meta_cols].copy()
for label, cols in bin_col_map.items():
    df_out[label] = df[cols].mean(axis=1)

# 保存
os.makedirs("../data/middle", exist_ok=True)
save_path = f"../data/middle/train_spectrum_bin{bin_width}.csv"
df_out.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"ビン幅: {bin_width} cm⁻¹ | ビン数: {len(bin_col_map)}")
print(f"shape: {df_out.shape}")
print(f"保存完了: {save_path}")


ビン幅: 1000 cm⁻¹ | ビン数: 6
shape: (1322, 11)
保存完了: ../data/middle/train_spectrum_bin1000.csv


In [3]:
df = pd.read_csv("../data/middle/train_spectrum_bin100.csv", encoding='utf-8-sig')
print(df.head())

   sample number  time  species number    樹種         含水率  4000-4100  \
0              1     1               1  イチョウ  216.129032   1.237228   
1              2     2               1  イチョウ  210.752688   1.207945   
2              3     3               1  イチョウ  205.913979   1.155000   
3              4     4               1  イチョウ  201.075269   1.104873   
4              5     5               1  イチョウ  196.236559   1.049295   

   4100-4200  4200-4300  4300-4400  4400-4500  ...  9000-9100  9100-9200  \
0   1.195882   1.144523   1.080303   1.026959  ...   0.398909   0.397366   
1   1.159597   1.111357   1.049931   0.999201  ...   0.404658   0.403267   
2   1.105197   1.059435   1.002155   0.954435  ...   0.395227   0.393965   
3   1.058732   1.014831   0.960605   0.915296  ...   0.386127   0.385018   
4   0.994718   0.954323   0.906272   0.865555  ...   0.374985   0.373957   

   9200-9300  9300-9400  9400-9500  9500-9600  9600-9700  9700-9800  \
0   0.395788   0.395013   0.395481   0.398043